[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traceopt-ai/traceml/blob/main/notebooks/data_loading_bottleneck.ipynb)

# The Half-Idle GPU: find a data-loading bottleneck in your own training run

Training jobs waste GPU time in a way nothing warns you about. The loss still descends, the run still finishes, and the GPU can sit half-idle the whole time, starved of data while the CPU decodes the next batch. This notebook lets you see that happen, and fix it, on your own hardware in a few minutes.

You will train a small ResNet **twice**, changing only the data loading, and read three things off each run: how long it took (wall clock), how busy the GPU was, and TraceML's one-line verdict. Between the two runs you change three DataLoader arguments, and watch the GPU get fed.

This is the runnable companion to the write-up [*The Half-Idle GPU*](https://medium.com/@apendyala/the-half-idle-gpu-40bbe394b834), where the same experiment on a 4-vCPU machine ran **1.8x faster** and flipped from input-bound to compute-bound. **How much you see here depends on your CPU cores.** A free Colab runtime has 2 vCPUs, so expect a real but smaller speedup with the GPU much better fed, and possibly still input-bound: two cores can only decode so fast. The final cell reads out the result for your specific hardware. That machine-dependence is the lesson, not a bug.

**Before you start:** switch to a GPU runtime via *Runtime -> Change runtime type -> Hardware accelerator: GPU* (a free T4 is exactly what the case study used), then run the cells top to bottom.

TraceML is open source: `pip install traceml-ai` (github.com/traceopt-ai/traceml).

## 1. Check the GPU

In [ ]:
!nvidia-smi -L
import torch

print("CUDA available:", torch.cuda.is_available())
assert (
    torch.cuda.is_available()
), "No GPU. Runtime -> Change runtime type -> GPU, then rerun."

## 2. Install TraceML
torch and torchvision come preinstalled on Colab; we only add TraceML. We pass `--no-deps` so pip does not try to downgrade Colab's numpy (TraceML's runtime deps are already present on Colab). On a fresh environment that lacks them, drop `--no-deps`.

In [ ]:
!pip install -q traceml-ai --no-deps

## 3. Get the data (full-resolution Imagenette, ~1.5 GB)
Full-resolution JPEGs are the point: decoding them is the heavy CPU cost that a single-worker loader cannot hide. (~1-2 min to download and extract.)

In [ ]:
import os

if not os.path.isdir("imagenette2/train"):
    !wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2.tgz
    !tar -xzf imagenette2.tgz
print("CPU cores:", os.cpu_count())
print("train classes:", len(os.listdir("imagenette2/train")))
print("train images:", sum(len(f) for _, _, f in os.walk("imagenette2/train")))

## 4. The training script
A standard ResNet-18 loop. The only TraceML additions are the two lines marked below: `traceml.init(...)` once, and a `with traceml.trace_step(model):` around the step. The `--profile` flag flips the data-loading settings and nothing else.

In [ ]:
%%writefile train.py
import argparse, os
import torch, torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import traceml_ai as traceml

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--profile", choices=["baseline", "optimized"],
                    default="baseline")
    ap.add_argument("--data-dir", default="imagenette2")
    ap.add_argument("--batch-size", type=int, default=64)
    ap.add_argument("--max-steps", type=int, default=300)
    args = ap.parse_args()

    optimized = args.profile == "optimized"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---- The ONLY thing that changes between the two profiles: data loading ----
    # Match workers to CPU cores so we do not oversubscribe (Colab free tier
    # has ~2 cores; more workers than cores thrash instead of overlapping).
    num_workers = min(4, os.cpu_count() or 2) if optimized else 0
    transform = T.Compose([
        T.RandomResizedCrop(224),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])
    dataset = torchvision.datasets.ImageFolder(
        os.path.join(args.data_dir, "train"), transform=transform)
    loader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=optimized,                              # async H2D staging
        persistent_workers=(optimized and num_workers > 0),
        drop_last=True,
    )
    # AMP is OFF in both profiles on purpose: it would speed up compute and
    # confound a demo whose whole point is isolating the data-loading change.

    model = torchvision.models.resnet18(
        weights=None, num_classes=len(dataset.classes)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    print(f"[demo] profile={args.profile} device={device} "
          f"num_workers={num_workers} batch={args.batch_size} "
          f"max_steps={args.max_steps} amp=off", flush=True)

    traceml.init(mode="auto")            # <-- TraceML line 1

    model.train()
    step = 0
    while step < args.max_steps:
        for images, labels in loader:
            with traceml.trace_step(model):   # <-- TraceML line 2
                images = images.to(device, non_blocking=optimized)
                labels = labels.to(device, non_blocking=optimized)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(images), labels)
                loss.backward()
                optimizer.step()
            step += 1
            if step >= args.max_steps:
                break


if __name__ == "__main__":
    main()


## 5. Run 1 - baseline (`num_workers=0`)
The default, single-process data loading. 300 steps (the verdict needs at least ~50; use more for steadier numbers). Watch the end-of-run summary: it should say **INPUT-BOUND**.

In [ ]:
!traceml run --mode summary --logs-dir logs --run-name baseline train.py --args --profile baseline --data-dir imagenette2 --max-steps 300 --batch-size 64

## 6. Run 2 - optimized (workers matched to CPU cores + pinned + persistent)
Same model, same data, same steps. Only the data loading changes (workers on, `pin_memory`, `persistent_workers`, non-blocking H2D). The GPU utilization should rise and the run should finish faster; if your machine has enough cores to fully hide the decode, the verdict flips to **COMPUTE-BOUND**.

In [ ]:
!traceml run --mode summary --logs-dir logs --run-name optimized train.py --args --profile optimized --data-dir imagenette2 --max-steps 300 --batch-size 64

## 7. Before vs after
The numbers below come from the **wall clock** (how long each run actually took) plus GPU and CPU utilization, read straight from each run's summary. The cell interprets the result for your specific hardware.

In [ ]:
import json


def load(run):
    d = json.load(open(f"logs/{run}/final_summary.json"))
    g = d["system"]["global"]["average"]
    return {
        "wall_s": round(d["duration_s"], 1),
        "verdict": d["primary_diagnosis"]["status"],
        "gpu_util": round(g["gpu_util_percent"], 1),
        "cpu_util": round(g["cpu_percent"], 1),
    }


b, o = load("baseline"), load("optimized")
speedup = b["wall_s"] / o["wall_s"]
saved = (1 - o["wall_s"] / b["wall_s"]) * 100

print(f"{'metric':<20}{'baseline':>15}{'optimized':>15}")
print("-" * 50)
print(f"{'wall clock (s)':<20}{b['wall_s']:>15}{o['wall_s']:>15}")
print(f"{'GPU util (%)':<20}{b['gpu_util']:>15}{o['gpu_util']:>15}")
print(f"{'CPU util (%)':<20}{b['cpu_util']:>15}{o['cpu_util']:>15}")
print(f"{'TraceML verdict':<20}{b['verdict']:>15}{o['verdict']:>15}")
print("-" * 50)
print(
    f"\nSpeedup (wall clock): {speedup:.2f}x  "
    f"({saved:.0f}% less GPU time for identical training)\n"
)

if o["verdict"] != "INPUT-BOUND":
    print(
        "The fix flipped the bottleneck: your machine had enough CPU to hide"
    )
    print("the decode, so the GPU is now the limit (COMPUTE-BOUND).")
else:
    print(
        f"Workers helped: GPU util {b['gpu_util']}% -> {o['gpu_util']}%, "
        f"{speedup:.2f}x faster."
    )
    print(f"But the CPU is now the ceiling ({o['cpu_util']}% used) and still")
    print("cannot fully feed the GPU, so it stays input-bound. With more CPU")
    print("cores this flips to COMPUTE-BOUND (the case study ran on a 4-core")
    print(
        "box: 1.8x, fully compute-bound). That is the machine-dependence the"
    )
    print("post is about, showing up live on your hardware.")

## What just happened
With `num_workers=0`, every batch of JPEGs is decoded in the training process while the GPU waits, then the GPU trains while the CPU waits. Turning on worker processes lets the decode happen in parallel, during time the GPU was going to spend computing anyway. The decode work did not get cheaper, it just stopped blocking the GPU.

**It depends on your machine.** This fix needs spare CPU cores to absorb the decode. On the 4-core box in the case study, workers fully hid the decode and the run went from INPUT-BOUND to COMPUTE-BOUND (~1.8x). On a 2-core Colab CPU the win is smaller and the job may stay input-bound, because 2 cores can only decode so fast (watch the CPU utilization hit ~100%). More cores, or lighter decode, widen the gap. We set `num_workers` to your core count to avoid oversubscribing them.

And it is not always the fix: on a job that is already compute-bound (try `resnet50` in the script), more workers buy nothing. Which of your jobs is which is exactly what the measurement tells you.

Try it on your own model: add the two TraceML lines to your training loop and run it under `traceml run`. github.com/traceopt-ai/traceml